# Hangman

Guess the hidden word one letter at a time. You have **6 mistakes** before the hangman is complete.

**How to play:** run the cells from top to bottom (*Run All*). The last cell starts the game; type a letter into the input box and press Enter. Repeated guesses and invalid input cost nothing.

**Word list:** put `words.txt` (one word per line) in the same folder as this notebook, e.g. the [google-10000-english list](https://raw.githubusercontent.com/first20hours/google-10000-english/master/google-10000-english-no-swears.txt). Without it, a small built-in list is used.

## 1. Setup

In [ ]:
import os
import random

try:
    from IPython.display import clear_output
except ImportError:          # running outside Jupyter
    clear_output = None


def clear_screen():
    """Clear the notebook cell output (or the terminal if not in Jupyter)."""
    if clear_output is not None:
        clear_output(wait=True)
    else:
        os.system("cls" if os.name == "nt" else "clear")


## 2. Game data: ASCII art and word list

In [ ]:
MAX_MISTAKES = 6

# Fallback word list, used when no .txt word list file is found.
BUILTIN_WORDS = [
    "python", "variable", "function", "keyboard", "terminal", "algorithm",
    "database", "compiler", "software", "network", "developer", "internet",
    "elephant", "mountain", "chocolate", "umbrella", "adventure", "universe",
    "treasure", "labyrinth", "notebook", "airplane", "festival", "gallows",
    "mystery", "journey", "kingdom", "library", "painting", "squirrel",
]

# One ASCII picture per number of mistakes (index 0 to 6).
STAGES = [
    """
      +---+
      |   |
          |
          |
          |
          |
    =========
    """,
    """
      +---+
      |   |
      O   |
          |
          |
          |
    =========
    """,
    """
      +---+
      |   |
      O   |
      |   |
          |
          |
    =========
    """,
    """
      +---+
      |   |
      O   |
     /|   |
          |
          |
    =========
    """,
    """
      +---+
      |   |
      O   |
     /|\\  |
          |
          |
    =========
    """,
    """
      +---+
      |   |
      O   |
     /|\\  |
     /    |
          |
    =========
    """,
    """
      +---+
      |   |
      O   |
     /|\\  |
     / \\  |
          |
    =========
    """,
]

VICTORY_ART = r"""
  __   __  ___   _   _    __        _____ _   _ _
  \ \ / / / _ \ | | | |   \ \      / /_ _| \ | | |
   \ V / | | | || | | |    \ \ /\ / / | ||  \| | |
    | |  | |_| || |_| |     \ V  V /  | || |\  |_|
    |_|   \___/  \___/       \_/\_/  |___|_| \_(_)
"""

DEFEAT_ART = r"""
   ____    _    __  __ _____    _____     _______ ____
  / ___|  / \  |  \/  | ____|  / _ \ \   / / ____|  _ \
 | |  _  / _ \ | |\/| |  _|   | | | \ \ / /|  _| | |_) |
 | |_| |/ ___ \| |  | | |___  | |_| |\ V / | |___|  _ <
  \____/_/   \_\_|  |_|_____|  \___/  \_/  |_____|_| \_\
"""


# The bundled word list is scraped from the web, so it contains some spam,
# adult, and programming-junk entries. Skip those so every puzzle is a fair,
# family-friendly English word.
BLOCKLIST = {
    "starsmerchant", "acdbentity", "ampland", "bufing", "pichunter",
    "thumbzilla", "voyeurweb", "voyuer", "thehun", "kijiji", "msgstr",
    "msgid", "sbjct", "postposted", "nutten", "titten", "verzeichnis",
    "uniprotkb", "twiki", "vbulletin", "jelsoft", "dicke", "gratuit",
    "transexuales", "transexual", "transsexual", "travesti", "zshops",
    "hydrocodone", "phentermine", "tramadol", "cialis", "levitra",
    "valium", "ambien", "adipex", "fioricet", "zoloft", "paxil", "xanax",
    "propecia", "prozac", "ultram", "spank", "spanking", "beastality",
    "shaved", "sucking", "peeing", "chubby", "lingerie", "fetish", "naked",
    "nudist", "erotica", "hairy", "booty", "lesbian", "lesbians", "mature",
    "deviant", "thongs", "thong", "pantyhose", "stockings", "livecam",
    "babes", "teens", "teen", "sexual", "sexuality", "casino", "casinos",
    "poker", "gambling", "blackjack", "roulette", "holdem", "porn",
}

MIN_WORD_LENGTH = 5


def load_words(path="words.txt"):
    """Load words from a text file (one per line); fall back to the built-in list."""
    candidates = [path]
    for candidate in candidates:
        if candidate and os.path.isfile(candidate):
            try:
                with open(candidate, encoding="utf-8") as f:
                    words = [
                        w for w in (line.strip().lower() for line in f)
                        if w.isascii() and w.isalpha()
                        and len(w) >= MIN_WORD_LENGTH and w not in BLOCKLIST
                    ]
                if words:
                    return words
            except OSError:
                pass
    return BUILTIN_WORDS


## 3. Game logic

In [ ]:
def masked_word(secret, guessed):
    """Return the word with unguessed letters hidden, e.g. 'P _ T H _ N'."""
    return " ".join(ch.upper() if ch in guessed else "_" for ch in secret)


def draw(secret, guessed, wrong, message=""):
    clear_screen()
    print("=" * 40)
    print("            H A N G M A N")
    print("=" * 40)
    print(STAGES[len(wrong)])
    print(f"  Word:    {masked_word(secret, guessed)}\n")
    print(f"  Wrong:   {' '.join(sorted(l.upper() for l in wrong)) or '-'}")
    print(f"  Guessed: {' '.join(sorted(l.upper() for l in guessed | wrong)) or '-'}")
    print(f"  Lives:   {'♥ ' * (MAX_MISTAKES - len(wrong))}"
          f"{'x ' * len(wrong)}({MAX_MISTAKES - len(wrong)} left)\n")
    if message:
        print(f"  >> {message}\n")


def get_guess(already_used):
    """Prompt until a valid single letter is entered. Returns (letter, message)."""
    raw = input("  Guess a letter: ").strip().lower()
    if len(raw) != 1 or not raw.isalpha() or not raw.isascii():
        return None, "Please enter a single letter (A-Z)."
    if raw in already_used:
        return None, f"You already guessed '{raw.upper()}'. No penalty, try another!"
    return raw, ""


def play_round(words):
    secret = random.choice(words)
    letters_needed = set(secret)
    guessed = set()   # correct letters
    wrong = set()     # incorrect letters
    message = "Good luck!"

    while True:
        draw(secret, guessed, wrong, message)

        if letters_needed <= guessed:
            print(VICTORY_ART)
            print(f"  You saved him! The word was '{secret.upper()}'.")
            print(f"  Mistakes made: {len(wrong)}/{MAX_MISTAKES}\n")
            return True

        if len(wrong) >= MAX_MISTAKES:
            print(DEFEAT_ART)
            print(f"  The word was '{secret.upper()}'. Better luck next time!\n")
            return False

        letter, message = get_guess(guessed | wrong)
        if letter is None:
            continue  # invalid or duplicate: no penalty

        if letter in secret:
            guessed.add(letter)
            message = f"Nice! '{letter.upper()}' is in the word."
        else:
            wrong.add(letter)
            message = f"Sorry, no '{letter.upper()}' in the word."


## 4. Play!

In [ ]:
def play_hangman(word_file="words.txt"):
    """Play rounds until you choose to stop. Type your guesses in the box that appears."""
    words = load_words(word_file)
    wins = losses = 0
    try:
        while True:
            if play_round(words):
                wins += 1
            else:
                losses += 1
            print(f"  Score  ->  Wins: {wins}   Losses: {losses}\n")
            if input("  Play again? (y/n): ").strip().lower() not in ("y", "yes"):
                break
    except KeyboardInterrupt:
        print()
    print("\n  Thanks for playing Hangman. Goodbye!\n")


play_hangman()
